Industry-Style Agentic RAG with LangGraph

Why this is industry-style

A simple RAG system searches only the private knowledge base.

An industry-style Agentic RAG system should do this:

Question

  ↓

Search private knowledge base first

  ↓

Grade the private evidence

  ↓

If private evidence is good → answer from KB

  ↓


If private evidence is weak → search the web

  ↓
Grade web evidence

  ↓

Generate a grounded answer with source type

This design is useful because private documents may be incomplete or outdated.



1.install the dependencies 

In [3]:
import os
from getpass import getpass
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass
if not os.getenv("GROQ_API_KEY"):
    os.ENVIRON["GROQ_API_KEY"]=getpass("enter the groq api key")

if not os.getenv("TAVILY_API_KEY"):
    os.ENVIRON["TAVILY_API_KEY"]=getpass("enter the tavily api key")

if not os.getenv("PINECONE_API_KEY"):
    os.ENVIRON["PINECONE_API_KEY"]=getpass("enter the pinecopne api key")

print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))


GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


3. Load Real Documents

We will create our private knowledge base from a real web document:


https://docs.langchain.com/oss/python/langgraph/agentic-rag


This becomes our internal/private KB for the demo.



The web fallback is separate. That means:


First source: indexed LangGraph documentation page

Fallback source: live web search through Tavily


In [4]:
from langchain_community.document_loaders import WebBaseLoader
SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"
loader=WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)
raw_docs=loader.load()
print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])


C:\Users\windows\AppData\Local\Temp\ipykernel_12708\591888355.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCa

4.split the documents into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    add_start_index=True
)
chunks=splitter.split_documents(raw_docs)
print("total chunks:",len(chunks))
print("\n first chunk preview:\n")
print(chunks[0].page_content[:900])

total chunks: 62

 first chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with


5. Create Free Local Embeddings

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings":True},

)
sample_embedding=embeddings.embed_query("what is agentic rag")
print("embedding deimensions:",len(sample_embedding))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3815.16it/s]


embedding deimensions: 384


6. Create the Pinecone Vector Database

The embedding model used in this notebook produces 384-dimensional vectors, so the Pinecone index must use dimension 384.

In [7]:
from pinecone import Pinecone,ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time
INDEX_NAME = "industry-agentic-rag-kb" #name of the database means index name 
NAMESPACE = "langgraph-agentic-rag" #name of the table means name space

#connect to pinecone 
pc=Pinecone(api_key=os.environ["PINECONE_API_KEY"])

#create the index only if it does not already exists
existing_index=[index_info["name"] for index_info in pc.list_indexes()] #list_indexes() is the function provided by pc

if INDEX_NAME not in existing_index:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
print("pinecone index ready:",INDEX_NAME)
#upload the document chunks and create the langchain vextor store 
vector_store=PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
retriever=vector_store.as_retriever(
    search_kwargs={
        "k":4,
        "namespace":NAMESPACE,
    }
)
print("pinecone vector store and retriever are ready")

pinecone index ready: industry-agentic-rag-kb
pinecone vector store and retriever are ready


7. Test Private KB Retrieval

In [8]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):#enumerate gives index as well as the real vlaues(doc) and 1 means start counting from 1 because by default it counts from 0 
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Fetch and preprocess documents for retrieval.
Index those documents for semantic search and create a retriever tool for the agent.
Build an agentic RAG system that can decide when to use the retriever tool.


​Concepts
This tutorial covers the following concepts:

Retrieval using

document loaders,
text splitters, embeddings, and
vector stores


The LangGraph Graph API, including state, nodes, edges, and conditional edges.

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs

8. Initialize Groq LLM

In [9]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

print(llm.invoke("Explain Agentic RAG in one sentence.").content)

Agentic RAG is a retrieval‑augmented generation paradigm in which the language model acts as an autonomous agent that dynamically decides when and how to query external knowledge sources and then integrates the retrieved information into its generated response.


9. Initialize Tavily Web Search

Tavily will be used only when the private knowledge base is insufficient.

This is important because we do not want to send every private/company question to the public web.

In [10]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

print("Tavily search tool ready.")

Tavily search tool ready.


10. Define Structured Decisions

We need reliable yes/no or route outputs from the LLM.

Structured outputs help avoid parsing messy natural language.

In [11]:
from typing import List,Literal,Optional
from pydantic import BaseModel,Field
from langchain_core.documents import Document
from typing_extensions import TypedDict

class RouteDecisiosn(BaseModel):
    route:Literal["retrieve","direct"]=Field(description="use kb for agentic rag docs and direct for simple chat/greetings")

class EvidenceGrade(BaseModel):
    grade:Literal["good","weak"]=Field(description="good means the evdence can answer the question and bad means we dont have enough evidence to answer the question")

class AgentState(TypedDict):
    question: str
    current_query: str
    kb_docs: List[Document]
    web_results: str
    kb_grade: str
    web_grade: str
    answer: str
    source_used: str
    retry_count: int